In [ ]:
# Phase 8 confirm: d3 -- a2's config (group attention over every (slot, group) token,
# five of nine windows drawn at random) trained on the three-source blend, the target
# Screen D promoted. Five folds, read at five and nine groups, scored as the seventh member.
import glob, os, shutil, sys, time

GIT_SHA = 'knee-phase11-screen-b-s0'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

from knee.dataset import PreppedSlotDataset, _SLOT_PLANES
from knee.dicom import SLOTS, SLOTS_V3
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import (differential_param_groups, make_folds, masked_bce_loss,
                        train_one_epoch)
# The stale-src guard, derived from what this run needs. Without packing the
# arms would differ in BatchNorm statistics as well as the table; without the
# v3 plane lookup the second arm dies on its first study.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel._slot_features', KneeModel._slot_features, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('PreppedSlotDataset.__init__', PreppedSlotDataset.__init__,
     ('weights_df', 'random_groups')),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, '_backbone_features'), 'rsna-knee-src predates slot packing'
assert all(name in _SLOT_PLANES for name, *_ in SLOTS_V3), 'rsna-knee-src cannot load v3 slots'
assert hasattr(KneeModel, '_group_tokens'), 'rsna-knee-src predates the group_attention head'
import knee.reports
assert hasattr(knee.reports, 'soft_bootstrap'), 'rsna-knee-src predates soft_bootstrap'
print('src carries slot packing, group attention and random_groups | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# One config, five folds. Each fold's model predicts its held-out fold and all 58
# gold studies; gold is excluded from training in every fold, so the five gold
# prediction sets average into an honest transfer number for the fold ensemble.
import math
assert 'accumulation_steps' in inspect.signature(train_one_epoch).parameters, (
    'rsna-knee-src predates gradient accumulation -- re-version it before pushing')
NAME = 'n1b_softboot50_s0'
BACKBONE = 'efficientnet_b0'
N_GROUPS, OUT_SIZE, EPOCHS, BATCH, HEAD = 5, 224, 8, 8, 'group_attention'
# c2's batch; one step per batch, as screened
ACCUMULATION = 1
# the stored windows, of which training draws N_GROUPS at random and evaluation takes
# the fixed spread (0,2,4,6,8 -- prep v2's pixels) or all nine
READS = (5, 9)
SOURCE = 'steven_v4'
HYPOTHESIS = ('Phase 11 N1 arm b: d3 config, fold 0 only, seed 0, target (1-a)*blend + a*teacher8 OOF with a=0.5. Was: Phase 8 confirm of d3 over all 5 folds of primary_v3: a2 (six slots, group-token '
              'attention, five of nine windows at random) trained on the probability blend of '
              'steven_v4, ours and lixin73, scored against steven_v4 binarised. Pooled OOF added '
              'to the six-member confirm OOF is the promotion metric.')
# the fixed truth for scoring, as every member has been scored
_, eval_labels = LABEL_SOURCES[SOURCE]
# the training frame Screen D promoted: steven_v4 + ours + lixin73, from the confidence
# cell's _votes (ours, steven_v4, pilkwang_v2, lixin73)
from knee.reports import blend_sources
assert len(_votes) == 4, f'expected four label sources, found {len(_votes)}'
_blend = blend_sources([_votes[1], _votes[0], _votes[3]], LABEL_COLUMNS)
_blend.index.name = 'StudyInstanceUID'
from knee.reports import soft_bootstrap
_teacher = pd.read_csv(glob.glob('/kaggle/input/**/teacher8_oof.csv', recursive=True)[0])
_teacher['StudyInstanceUID'] = _teacher['StudyInstanceUID'].astype(str)
_teacher = _teacher.set_index('StudyInstanceUID')[LABEL_COLUMNS]
assert len(_teacher) == 4349, len(_teacher)
_mixed = soft_bootstrap(_blend, _teacher, alpha=0.5, forbidden_uids=holdout)
_mixed.index.name = 'StudyInstanceUID'
_moved = (_mixed - _blend.reindex(_mixed.index)).abs().stack().mean()
print(f'soft bootstrap: mean |target change| {_moved:.4f}')
train_labels = _mixed.reset_index()
print(f'blend target: {len(train_labels)} studies, positive rate per label '
      f'{train_labels[LABEL_COLUMNS].mean().min():.3f}-{train_labels[LABEL_COLUMNS].mean().max():.3f}')

EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = 'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,macro_auc,paired_delta,train_minutes,inference_seconds,promoted,gold_macro'
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

def loader(uids, labels_df, shuffle=False, augment=False, random_groups=False,
           n_groups=N_GROUPS):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=n_groups,
                            out_size=OUT_SIZE, slots=slot_names, augment=augment,
                            random_groups=random_groups)
    # the 9-group artifacts decode 27 slices a slot whatever is read, so the loader gets
    # every core the session has
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=4)

labeled_uids = [u for u in all_uids if u not in holdout and u in uid_to_npz]
row_of = {u: i for i, u in enumerate(labeled_uids)}
oof_true = np.full((len(labeled_uids), len(LABEL_COLUMNS)), np.nan)
oof_pred = np.full_like(oof_true, np.nan)
oof_pred9 = np.full_like(oof_true, np.nan)
gold_by_fold, gold_by_fold9 = [], []
print(f'{NAME}: {len(labeled_uids)} labelled studies across 5 folds')

t_start = time.time()
torch.manual_seed(0); np.random.seed(0)
for fold in (0,):
    tr_uids, va_uids = train_val_split(folds, val_fold=fold, exclude_uids=holdout)
    tr_uids = [u for u in tr_uids if u in uid_to_npz]
    va_uids = [u for u in va_uids if u in uid_to_npz]

    train_loader = loader(tr_uids, train_labels, shuffle=True, augment=True,
                          random_groups=True)
    model = KneeModel(backbone_name=BACKBONE, num_labels=12, pretrained=True, head=HEAD).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-4, epochs=EPOCHS,
        steps_per_epoch=math.ceil(len(train_loader) / ACCUMULATION))

    t0 = time.time()
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, device=device,
                               scaler=scaler, scheduler=scheduler,
                               accumulation_steps=ACCUMULATION)
    minutes = (time.time() - t0) / 60

    y_true, y_pred = evaluate(model, loader(va_uids, eval_labels), device=device)
    for uid, yt, yp in zip(va_uids, y_true, y_pred):
        oof_true[row_of[uid]] = yt
        oof_pred[row_of[uid]] = yp
    aucs = per_label_auc(y_true, y_pred)
    fold_macro = macro_auc(y_true, y_pred)

    g_true, g_pred = evaluate(model, loader(gold_uids, gold_labels), device=device)
    gold_by_fold.append(g_pred)
    # the same checkpoint reading all nine windows, in this session rather than a second one
    _, y_pred9 = evaluate(model, loader(va_uids, eval_labels, n_groups=9), device=device)
    for uid, yp in zip(va_uids, y_pred9):
        oof_pred9[row_of[uid]] = yp
    _, g_pred9 = evaluate(model, loader(gold_uids, gold_labels, n_groups=9), device=device)
    gold_by_fold9.append(g_pred9)
    moved = float(np.abs(y_pred9 - y_pred).max())
    assert moved > 1e-4, 'the nine-group read returned the five-group predictions'
    print(f'fold {fold}: val macro {fold_macro:.4f} (nine groups '
          f'{macro_auc(y_true, y_pred9):.4f}), gold {macro_auc(g_true, g_pred):.4f} '
          f'(nine {macro_auc(g_true, g_pred9):.4f}), final loss {loss:.4f}, {minutes:.1f} min '
          f'({(time.time() - t_start) / 3600:.1f}h elapsed, largest read change {moved:.2f})')
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=NAME,
                   gold_macro=float(macro_auc(g_true, g_pred)),
                   hypothesis=HYPOTHESIS, fold_set=f'primary_v3_fold{fold}', seed=fold,
                   per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(fold_macro), paired_delta=float('nan'),
                   train_minutes=minutes, inference_seconds=float('nan'), promoted=False)
    torch.save(model.state_dict(), f'/kaggle/working/{NAME}_fold{fold}.pt')
    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

_rows = ~np.isnan(oof_pred).any(axis=1)
assert _rows.sum() == len(va_uids), 'fold 0 rows do not match its validation set'
pd.DataFrame({'StudyInstanceUID': [u for u, r in zip(labeled_uids, _rows) if r]}).to_csv(
    f'/kaggle/working/val_uids_{NAME}.csv', index=False)
np.save(f'/kaggle/working/val_pred_{NAME}_g5.npy', oof_pred[_rows])
np.save(f'/kaggle/working/val_pred_{NAME}_g9.npy', oof_pred9[_rows])
np.save(f'/kaggle/working/val_true_{NAME}.npy', oof_true[_rows])
np.save(f'/kaggle/working/gold_pred_{NAME}_g5.npy', gold_by_fold[0])
np.save(f'/kaggle/working/gold_pred_{NAME}_g9.npy', gold_by_fold9[0])
np.save('/kaggle/working/gold_true.npy', g_true)
train_labels.to_csv(f'/kaggle/working/target_{NAME}.csv', index=False)
print(f'saved fold-0 reads for {NAME}; total {(time.time() - t_start) / 3600:.2f}h')
